# The bug that costs a quarter, and the type that makes it impossible

A number in a model is never just a float. It is *so many dollars per week*, or *so much
outcome per unit of dose*, and the two most expensive mistakes in applied causal work are
both category errors that a float will happily commit:

1. **A rate added to a total.** A per-week effect summed into a cumulative one. The model
   fits, the residuals look fine, and the answer is off by the number of weeks.
2. **A unit swapped for its cousin.** USD read as EUR, a daily series joined to a weekly
   one. Nothing raises. The report is wrong by ten percent and stays wrong.

Neither is caught by tests, because both produce numbers of entirely plausible size. The
parent repo grew a whole `finance/` package trying to keep them apart by convention.

`axiom.core` keeps three things distinct instead, in the type system:

| | Example | On mismatch |
|---|---|---|
| **Dimension** | currency vs time vs outcome | raise, always |
| **Unit** | USD vs EUR; day vs week | convert if a conversion is registered, with a ledger line; raise if not |
| **Scope** | this population, this window | never auto-resolved; needs a stated assumption (`TransferPlan`, Phase 6) |

This notebook covers the first two rows.

In [ ]:
from fractions import Fraction

import numpy as np

from axiom.core import (
    BASES,
    D,
    UNITS,
    BaseRegistry,
    Dimension,
    DimensionError,
    UndeclaredBaseError,
    UnitConversionError,
    UnitSystem,
    dimensionless,
)

from axiom.display import enable, table

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import CRITICAL, ORANGE, annotate, caption, compare, lines

enable();  # every axiom result renders itself from here on

## What the mistake actually costs

Here is the first failure, drawn. A weekly rate of decay applied as though it were a daily
one: the same code, the same units of "per period", one line of the model reading the wrong
column. Both curves are smooth, both are plausible, and one of them is off by a factor of
seven at the end of the quarter.

In [ ]:
weeks = np.linspace(0.0, 13.0, 60)
rate_per_week = 0.18
correct = 100.0 * np.exp(-rate_per_week * weeks)
mistaken = 100.0 * np.exp(-rate_per_week * weeks * 7.0)   # the rate was per *day*

fig = lines(
    weeks,
    {"rate read as per-week": correct, "rate read as per-day": mistaken},
    title="Nothing in a float says which period it is per",
    subtitle="one decay rate, two readings of the same number — neither raises, neither looks wrong",
    x_title="weeks", y_title="carryover remaining (%)",
)
annotate(fig, 13.0, correct[-1], f"{correct[-1]:.0f}% vs {mistaken[-1]:.2f}%")
caption(fig, "This is the failure D.time exists to make unrepresentable: the two quantities "
             "have different dimensions, so the sum that produced the second one cannot be built.")

## The base set is declarable

`BASES` ships four bases — `time`, `currency`, `outcome`, `entity` — and a domain declares
whatever else it needs. `D` is the same registry; `D.currency` reads better in model code.

In [ ]:
print(list(BASES))
print(D.currency, D.time, D.outcome, D.entity)

In [ ]:
mass = BASES.declare("mass", symbol="M")   # agronomy, pharmacology
print(mass, "|", BASES.declarations())     # non-default declarations travel with a saved analysis

A base that has not been declared is an error naming the base, not a silent new dimension.
The alternative — inventing a dimension on first use — means a typo (`temperture`) becomes a
new physical quantity that is incompatible with everything, including itself spelled right.

In [ ]:
try:
    Dimension(exponents={"temperature": 1})
except UndeclaredBaseError as e:
    print(type(e).__name__, "->", e)

## Algebra

Exponents are `Fraction`s, not ints, because a standard deviation is the square root of a
variance and intervals are everywhere in this package. An integer exponent would make
`sqrt(variance)` either an error or a lie. Zero exponents vanish, so equality and hashing
are structural.

In [ ]:
outcome_per_dose = D.outcome / D.currency
rate = D.outcome / D.time
variance = outcome_per_dose**2

print("outcome per unit dose:", outcome_per_dose)
print("its variance:         ", variance)
print("sd = sqrt(variance):  ", variance.root(2), "==", outcome_per_dose, "->", variance.root(2) == outcome_per_dose)
print("x / x is dimensionless:", (rate / rate) == dimensionless(), (rate / rate).is_dimensionless)
print("fractional powers:     ", D.currency ** Fraction(1, 3))

`require_equal` and `require_dimensionless` are what the expression-tree checker (Phase 1b)
calls at every node; the error names both sides and the node it was checking. An error that
says only "shape mismatch" sends a reader to the array; one that names the node sends them to
the line of the model that is wrong.

In [ ]:
try:
    D.currency.require_equal(D.time, context="Add(term_1, term_2)")
except DimensionError as e:
    print(e)

try:
    D.currency.require_dimensionless(context="the argument of log()")
except DimensionError as e:
    print(e)

## A `Dimension` is a `Spec`

So it serializes, hashes, and round-trips like everything else (see `02-specs-and-hashing`).
A dimension that could not be saved would mean a reloaded analysis had to re-derive its own
units — and would be free to derive them differently.

In [ ]:
s = outcome_per_dose.to_json()
print(s)
print(Dimension.from_json(s) == outcome_per_dose, outcome_per_dose.content_hash()[:16])

## Units and conversions

A `UnitSystem` attaches units of measure to bases and registers conversions *within* a
dimension. `UNITS` is the process-global instance; you can also build a private one.
Conversions compose: registering USD→EUR and EUR→GBP makes USD→GBP available, so a currency
graph needs `n − 1` registered edges rather than `n²`.

In [ ]:
UNITS.declare("USD", "currency")
UNITS.declare("EUR", "currency")
UNITS.declare("GBP", "currency")
UNITS.declare("day", "time")
UNITS.declare("week", "time")

UNITS.register("USD", "EUR", Fraction(9, 10))
UNITS.register("EUR", "GBP", Fraction(17, 20))
UNITS.register("week", "day", 7)

print(UNITS.factor("USD", "GBP"), "|", UNITS.dimension_of("week"))

Every conversion performed returns the value **and** a `LedgerLine`. That line is the
provenance rule (CLAUDE.md rule 4) made concrete: the analysis ledger records that a number
crossed a unit boundary and by what factor, so the question "why is this figure 15% below
the one in the deck?" has an answer that is not archaeology.

In [ ]:
value, line = UNITS.convert(1000.0, "USD", "GBP")
print(value)
print(line.kind, "|", line.statement)
print(line.detail)

## The second failure, drawn

A budget reported in the wrong currency is not a rounding difference. Below: the same
1,000 units of spend under the three readings available to a float — and the fact that the
middle bar is *plausible* is exactly why nothing catches it.

In [ ]:
readings = {
    "as declared (USD)": 1000.0,
    "read as EUR, unconverted": UNITS.convert(1000.0, "USD", "EUR")[0],
    "read as GBP, unconverted": UNITS.convert(1000.0, "USD", "GBP")[0],
}
fig = compare(
    list(readings), list(readings.values()),
    highlight="as declared (USD)",
    value_fmt="{:,.0f}",
    title="Three answers, one float",
    subtitle="the same reported spend under three readings of an undeclared unit",
    x_title="reported spend",
)
caption(fig, "A 235-unit gap between the top and bottom bar, and no test in the repository "
             "would fail. The unit system turns the gap into either a conversion with a ledger "
             "line, or a refusal.")

Cross-dimension conversion is refused (that is the *dimension* row of the table), and a
missing conversion is an error rather than a guess (the *unit* row). The distinction matters:
the first is a modelling mistake, the second is a missing declaration, and telling a user
which one they have is most of the fix.

In [ ]:
UNITS.declare("JPY", "currency")

refused = []
for src, dst in [("USD", "day"), ("USD", "JPY")]:
    try:
        UNITS.convert(1.0, src, dst)
    except (DimensionError, UnitConversionError) as e:
        refused.append([f"{src} -> {dst}", type(e).__name__, str(e)])
table(refused, headers=("conversion", "raised", "why"))

A private registry and unit system are useful in tests and in adapters that should not
touch the global declarations — an adapter for one client's data should not be able to
change what `USD` means for everybody else in the process.

In [ ]:
private_bases = BaseRegistry()
private_units = UnitSystem()
private_units.declare("kg", "mass")
private_units.declare("g", "mass")
private_units.register("kg", "g", 1000)
print(private_units.convert(2.5, "kg", "g")[0], "|", private_units.units(), "|", private_units.conversions())
print("private registry knows only the defaults:", list(private_bases))

## What this bought you

Two whole classes of wrong number are now unrepresentable rather than untested: a rate
cannot be added to a total, and a currency cannot become another currency without a
recorded factor. Every number that *did* cross a boundary carries a ledger line saying so.

Everything above this layer inherits it for free — the expression tree checks a model's
dimensions at every node (`03-expression-tree`), an estimand states the population and
window it holds for (`nbs/estimands/`), and a transfer between them is refused unless the
assumption is written down (`nbs/calibrate/04-transfer-and-ledger.ipynb`).